# Building a CNN in PyTorch

_PyTorch (a complete course)_

**Wire conv, pool, and a linear head into a small image classifier, then train it.**

A CNN (Convolutional Neural Network) is a stack of layers that turn an image into a class.

       The first layers are convolutional blocks: Conv2d finds local patterns, BatchNorm2d steadies the signal, ReLU adds a nonlinearity, and MaxPool2d shrinks the map. Each block makes the picture smaller in space but richer in channels.

---

This notebook is a practice scaffold for the **Building a CNN in PyTorch** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- data: normalize, then a DataLoader feeds batches of (N, C, H, W) ---
tfm = transforms.Compose([
    transforms.ToTensor(),                       # uint8 0..255 -> float 0..1, shape (1, 28, 28)
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST mean/std -> zero mean, unit variance
])
train_ds = datasets.MNIST(root="./data", train=True,  download=True, transform=tfm)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=256, shuffle=False)

# --- model: 2 conv blocks + a Linear head (LeNet-style) ---
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),  # (N,1,28,28) -> (N,16,28,28)
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),                                       # -> (N,16,14,14)
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1), # -> (N,32,14,14)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                                       # -> (N,32,7,7)
        )
        # flattened size = channels * H * W = 32 * 7 * 7 = 1568
        self.head = nn.Linear(32 * 7 * 7, num_classes)            # outputs raw logits (no softmax!)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)   # keep batch dim, flatten the rest -> (N, 1568)
        return self.head(x)

model = SmallCNN().to(device)
loss_fn = nn.CrossEntropyLoss()                 # expects logits + integer class labels
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# --- training loop ---
EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()                               # BatchNorm uses batch stats now
    running = 0.0
    for images, labels in train_dl:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()                   # clear grads (they accumulate otherwise)
        logits = model(images)                  # forward
        loss = loss_fn(logits, labels)
        loss.backward()                         # backprop fills .grad
        optimizer.step()                        # update weights
        running += loss.item()                  # .item() detaches -> no graph kept across steps
    print(f"epoch {epoch+1}/{EPOCHS}  avg train loss {running/len(train_dl):.4f}")

# --- evaluation ---
model.eval()                                    # BatchNorm switches to running stats
correct = total = 0
with torch.no_grad():                           # no graph at inference -> less memory
    for images, labels in test_dl:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"test accuracy: {correct/total:.4f}  on {total} images")

## Visualize the data & results

_On the same image task, does a CNN really beat a plain MLP on flattened pixels?_

In [ ]:
import numpy as np, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0); np.random.seed(0)
d = load_digits()                                  # 1797 real 8x8 digits
X = (d.images / 16.0).astype("float32")            # scale pixels to 0..1, shape (N, 8, 8)
y = d.target.astype("int64")
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

# add channel axis -> (N, 1, 8, 8); PyTorch convs want (N, C, H, W)
Xtr_t = torch.tensor(Xtr).unsqueeze(1); Xte_t = torch.tensor(Xte).unsqueeze(1)
ytr_t = torch.tensor(ytr);              yte_t = torch.tensor(yte)
dl = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=64, shuffle=True)

class CNN(nn.Module):                              # two conv blocks + Linear head
    def __init__(s):
        super().__init__()
        s.c1 = nn.Conv2d(1, 8, 3, padding=1);  s.b1 = nn.BatchNorm2d(8)
        s.c2 = nn.Conv2d(8, 16, 3, padding=1); s.b2 = nn.BatchNorm2d(16)
        s.pool = nn.MaxPool2d(2); s.relu = nn.ReLU(); s.fc = nn.Linear(16*2*2, 10)
    def forward(s, x):
        x = s.pool(s.relu(s.b1(s.c1(x))))          # 8x8 -> 4x4
        x = s.pool(s.relu(s.b2(s.c2(x))))          # 4x4 -> 2x2
        return s.fc(x.flatten(1))                   # 16*2*2 = 64 -> 10 logits

class MLP(nn.Module):                              # flat pixels -> dense layers
    def __init__(s):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))
    def forward(s, x): return s.net(x.flatten(1))

def train(model, epochs=12):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); lf = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for xb, yb in dl:
            opt.zero_grad(); lf(model(xb), yb).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        return (model(Xte_t).argmax(1) == yte_t).float().mean().item()

torch.manual_seed(0); cnn_acc = train(CNN())
torch.manual_seed(0); mlp_acc = train(MLP())
print(round(cnn_acc, 4), round(mlp_acc, 4))        # -> 0.963 0.9315

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** An MNIST batch arrives as a NumPy array of shape (64, 28, 28). What must you do before the first Conv2d, and what is the target shape?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note PyTorch convolutions want (N, C, H, W). — _Conv2d expects an explicit channel axis even for grayscale._
- Add a channel axis: x = torch.tensor(arr).unsqueeze(1). — _That inserts C=1 at position 1, giving (64, 1, 28, 28)._

**Answer:** Convert to a tensor and add the channel dimension with unsqueeze(1), producing shape (64, 1, 28, 28). Feeding (64, 28, 28) straight in is a shape error.

</details>

**Problem 2.** A net does Conv(p=1, 3×3) → Pool2 → Conv(p=1, 3×3) → Pool2 on a 1×32×32 image, ending with 64 channels. What is the input size of the first Linear layer?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Padded 3×3 convs keep the spatial size; each Pool2 halves it. — _H_out = (H + 2p − k)/s + 1 = H for p=1,k=3,s=1; pool halves._
- 32 → (pool) 16 → (pool) 8, with 64 channels. — _Two MaxPool2d(2) layers halve twice: 32 → 16 → 8._
- Flattened size = 64 × 8 × 8. — _channels × height × width of the final map._

**Answer:** $64 \times 8 \times 8 = 4096$. So the head is Linear(4096, num_classes).

</details>

**Problem 3.** Test accuracy is much lower and noisier than your training accuracy suggested. The model uses BatchNorm. What is the most likely one-line bug?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check whether model.eval() is called before the test loop. — _BatchNorm uses batch stats in train mode and running stats in eval mode._
- Also wrap inference in torch.no_grad(). — _Saves memory and signals you are not training._

**Answer:** You forgot model.eval(), so BatchNorm normalized test batches with their own statistics instead of the stored running stats. Call model.eval() (and use torch.no_grad()) before evaluating.

</details>